In [ ]:
#Importacion de librerias
import requests
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve, precision_recall_curve
import xgboost as xgb
import joblib  # Para guardar y cargar el modelo
import shap    # Para la interpretabilidad del modelo


In [ ]:
# Leer un Excel a Python
df_importado = pd.read_excel("Despliegue2024.xlsx")

df_importado

In [ ]:

# ===========================
# 3️⃣ Separar columnas de identificación
# ===========================
columnas_identificacion = ['IDENTIFICACION', 'PERIODO', 'AÑO']
df_identificacion = df_importado[columnas_identificacion].copy()


In [ ]:

# ===========================
# 4️⃣ Preparar las features para el pipeline
# ===========================
# Columnas que no se usan como features
columnas_a_eliminar = columnas_identificacion + ['TARGET_DESERCION']

# DataFrame solo con features
X_nuevos = df_importado.drop(columns=columnas_a_eliminar, errors='ignore')


In [ ]:

# ===========================
# 5️⃣ Cargar el pipeline
# ===========================
pipeline = joblib.load('modelo_final.pkl')
print(f"Umbral guardado en el pipeline: {pipeline.best_threshold:.2f}")


In [ ]:

# ===========================
# 6️⃣ Asegurarse que columnas coincidan
# ===========================
X_nuevos = X_nuevos[pipeline.named_steps['preprocessor'].transformers_[0][2] + 
                    pipeline.named_steps['preprocessor'].transformers_[1][2]]


In [ ]:

# ===========================
# 7️⃣ Realizar predicciones
# ===========================
probabilidades = pipeline.predict_proba(X_nuevos)[:, 1]
predicciones = (probabilidades >= pipeline.best_threshold).astype(int)


In [ ]:

# ===========================
# 8️⃣ Combinar resultados con columnas de identificación
# ===========================
resultado_final = pd.concat([df_identificacion.reset_index(drop=True),
                             pd.DataFrame({
                                 'PROBABILIDAD_DESERCION': probabilidades,
                                 'PREDICCION_DESERCION': predicciones
                             })], axis=1)

print("Predicciones generadas con identificación:")
print(resultado_final.head())

In [ ]:
resultado_final

In [ ]:
df_importado.columns

In [ ]:
# Columnas a conservar
columnas_conservar = ['IDENTIFICACION', 'PERIODO', 'AÑO', 'TARGET_DESERCION']

# Crear un nuevo DataFrame solo con esas columnas
df_reducido = df_importado[columnas_conservar].copy()

print("DataFrame reducido:")
print(df_reducido.head())

In [ ]:
df5 = resultado_final.merge(
    df_reducido,
    on=['IDENTIFICACION', 'PERIODO', 'AÑO'],
    how='left'
)

In [ ]:
df5

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = df5['TARGET_DESERCION']
y_pred = df5['PREDICCION_DESERCION']

print("Reporte de Clasificación:\n", classification_report(y_true, y_pred))
print("Matriz de Confusión:\n", confusion_matrix(y_true, y_pred))

In [ ]:
plt.hist(df5['PROBABILIDAD_DESERCION'], bins=20, edgecolor='black')
plt.title("Distribución de Probabilidades de Deserción")
plt.xlabel("Probabilidad")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
print(f"Umbral aplicado: {pipeline.best_threshold:.2f}")

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

y_true = df5['TARGET_DESERCION']
y_scores = df5['PROBABILIDAD_DESERCION']

precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
auc_pr = auc(recall, precision)

plt.plot(recall, precision, marker='.')
plt.title(f"Curva Precision-Recall (AUC={auc_pr:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()